In [ ]:
# ============================================================
# WORLD REAL ESTATE DATASET (147K)
# FULL ETL + NAIVE BAYES CLASSIFICATION PROJECT
# FULLY CORRECT VERSION
# ============================================================


# ============================================================
# STEP 1 — IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# Train Test Split
from sklearn.model_selection import train_test_split

# Scaling
from sklearn.preprocessing import StandardScaler

# Encoding
from sklearn.preprocessing import LabelEncoder

# Naive Bayes Model
from sklearn.naive_bayes import GaussianNB

# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)

# Save Model
import joblib


# ============================================================
# STEP 2 — LOAD DATASET
# ============================================================

# Load dataset

df = pd.read_csv(
    "/content/world_real_estate_data(147k).csv"
)

print("DATASET LOADED SUCCESSFULLY")

# Show first 5 rows

print(df.head())


# ============================================================
# STEP 3 — COLUMN NAMES
# ============================================================

print("\n================ COLUMN NAMES ================")

print(df.columns)


# ============================================================
# STEP 4 — DATASET INFORMATION
# ============================================================

print("\n================ DATASET SHAPE ================")

print(df.shape)

print("\n================ DATASET INFO ================")

print(df.info())

print("\n================ MISSING VALUES ================")

print(df.isnull().sum())


# ============================================================
# STEP 5 — DROP UNNECESSARY COLUMNS
# ============================================================

# Remove unnecessary columns

drop_cols = ['title', 'image', 'url']

df.drop(
    columns=drop_cols,
    inplace=True,
    errors='ignore'
)

print("\n================ REMAINING COLUMNS ================")

print(df.columns)


# ============================================================
# STEP 6 — CONVERT AREA COLUMNS TO NUMERIC
# ============================================================

# Convert:
# "120 m²" -> 120

# apartment_total_area

df['apartment_total_area'] = (
    df['apartment_total_area']
    .astype(str)
    .str.replace(' m²', '', regex=False)
)

df['apartment_total_area'] = pd.to_numeric(
    df['apartment_total_area'],
    errors='coerce'
)


# apartment_living_area

df['apartment_living_area'] = (
    df['apartment_living_area']
    .astype(str)
    .str.replace(' m²', '', regex=False)
)

df['apartment_living_area'] = pd.to_numeric(
    df['apartment_living_area'],
    errors='coerce'
)

print("\n================ AREA COLUMNS CONVERTED ================")


# ============================================================
# STEP 7 — HANDLE MISSING VALUES
# ============================================================

# Numerical Columns

num_cols = df.select_dtypes(
    include=np.number
).columns

# Fill numerical missing values using median

for col in num_cols:
    df[col] = df[col].fillna(
        df[col].median()
    )


# Categorical Columns

cat_cols = df.select_dtypes(
    include='object'
).columns

# Fill categorical missing values using mode

for col in cat_cols:
    df[col] = df[col].fillna(
        df[col].mode()[0]
    )

print("\n================ MISSING VALUES AFTER CLEANING ================")

print(df.isnull().sum())


# ============================================================
# STEP 8 — CREATE TARGET COLUMN
# ============================================================

# Classification:
#
# 1 = Expensive Property
# 0 = Cheap Property

median_price = df['price_in_USD'].median()

df['target'] = (
    df['price_in_USD'] > median_price
).astype(int)

print("\n================ TARGET DISTRIBUTION ================")

print(df['target'].value_counts())


# ============================================================
# STEP 9 — REMOVE ORIGINAL PRICE COLUMN
# ============================================================

df.drop(
    columns=['price_in_USD'],
    inplace=True
)

print("\n================ DATA AFTER TARGET CREATION ================")

print(df.head())


# ============================================================
# STEP 10 — ENCODE CATEGORICAL DATA
# ============================================================

# Convert text into numbers

le = LabelEncoder()

for col in cat_cols:
    df[col] = le.fit_transform(
        df[col].astype(str)
    )

print("\n================ ENCODED DATASET ================")

print(df.head())


# ============================================================
# STEP 11 — FEATURES & TARGET
# ============================================================

# Features
X = df.drop('target', axis=1)

# Target
y = df['target']

print("\n================ FEATURE SHAPE ================")

print(X.shape)

print("\n================ TARGET SHAPE ================")

print(y.shape)


# ============================================================
# STEP 12 — TRAIN TEST SPLIT
# ============================================================

# 80% Train
# 20% Test

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\n================ TRAINING DATA ================")

print(X_train.shape)

print("\n================ TESTING DATA ================")

print(X_test.shape)


# ============================================================
# STEP 13 — FEATURE SCALING
# ============================================================

# Naive Bayes works better with scaling

scaler = StandardScaler()

# Fit + Transform training data

X_train_scaled = scaler.fit_transform(
    X_train
)

# Transform testing data

X_test_scaled = scaler.transform(
    X_test
)

print("\n================ SCALING COMPLETED ================")


# ============================================================
# STEP 14 — BUILD NAIVE BAYES MODEL
# ============================================================

# GaussianNB:
# Used for continuous numerical data

nb_model = GaussianNB()

# Train model

nb_model.fit(
    X_train_scaled,
    y_train
)

print("\n================ NAIVE BAYES MODEL TRAINED ================")


# ============================================================
# STEP 15 — PREDICTION
# ============================================================

# Predict testing data

y_pred = nb_model.predict(
    X_test_scaled
)

print("\n================ SAMPLE PREDICTIONS ================")

print(y_pred[:20])


# ============================================================
# STEP 16 — ACCURACY SCORE
# ============================================================

accuracy = accuracy_score(
    y_test,
    y_pred
)

print("\n================ ACCURACY ================")

print("Accuracy:", accuracy)


# ============================================================
# STEP 17 — CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    y_pred
)

print("\n================ CONFUSION MATRIX ================")

print(cm)


# ============================================================
# STEP 18 — CLASSIFICATION REPORT
# ============================================================

print("\n================ CLASSIFICATION REPORT ================")

print(
    classification_report(
        y_test,
        y_pred
    )
)


# ============================================================
# STEP 19 — CONFUSION MATRIX HEATMAP
# ============================================================

plt.figure(figsize=(6,5))

sns.heatmap(
    cm,
    annot=True,
    fmt='d'
)

plt.title("Naive Bayes Confusion Matrix")

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()


# ============================================================
# STEP 20 — SAVE MODEL
# ============================================================

joblib.dump(
    nb_model,
    "real_estate_naive_bayes.pkl"
)

joblib.dump(
    scaler,
    "scaler.pkl"
)

print("\n================ MODEL SAVED ================")


# ============================================================
# STEP 21 — LOAD MODEL
# ============================================================

loaded_model = joblib.load(
    "real_estate_naive_bayes.pkl"
)

loaded_scaler = joblib.load(
    "scaler.pkl"
)

print("\n================ MODEL LOADED ================")


# ============================================================
# STEP 22 — SINGLE PREDICTION
# ============================================================

# Take one sample row

sample_data = X_test.iloc[0:1]

# Scale sample data

sample_scaled = loaded_scaler.transform(
    sample_data
)

# Predict

prediction = loaded_model.predict(
    sample_scaled
)

print("\n================ SINGLE PREDICTION ================")

print(prediction)

# Meaning:
#
# 0 = Cheap Property
# 1 = Expensive Property


# ============================================================
# STEP 23 — PREDICTION PROBABILITY
# ============================================================

probability = loaded_model.predict_proba(
    sample_scaled
)

print("\n================ PREDICTION PROBABILITY ================")

print(probability)

# Example:
#
# [[0.35 0.65]]
#
# 35% Cheap
# 65% Expensive


# ============================================================
# STEP 24 — COUNTRY ANALYSIS
# ============================================================

country_analysis = df.groupby(
    "country"
)["target"].mean()

print("\n================ COUNTRY ANALYSIS ================")

print(country_analysis.head(10))


# ============================================================
# STEP 25 — TOP COUNTRIES VISUALIZATION
# ============================================================

top_country = country_analysis.sort_values(
    ascending=False
).head(10)

plt.figure(figsize=(12,6))

top_country.plot(kind='bar')

plt.title(
    "Top Countries with Expensive Properties"
)

plt.xlabel("Country")

plt.ylabel("Expensive Property Ratio")

plt.show()


# ============================================================
# STEP 26 — CORRELATION HEATMAP
# ============================================================

plt.figure(figsize=(14,10))

sns.heatmap(
    df.corr(numeric_only=True),
    annot=True
)

plt.title("Correlation Heatmap")

plt.show()


# ============================================================
# STEP 27 — NAIVE BAYES THEORY
# ============================================================

# Naive Bayes:
#
# Based on Bayes Theorem
#
# Assumes:
# all features are independent
#
# Fast and efficient algorithm
#
# Good for:
# - Spam Detection
# - Text Classification
# - Recommendation Systems


# ============================================================
# END OF PROJECT
# ============================================================

DATASET LOADED SUCCESSFULLY
                                               title   country  \
0  2 room apartment 120 m² in Mediterranean Regio...    Turkey   
1              4 room villa 500 m² in Kalkan, Turkey    Turkey   
2          1 room apartment 65 m² in Antalya, Turkey    Turkey   
3             1 room apartment  in Pattaya, Thailand  Thailand   
4             2 room apartment  in Pattaya, Thailand  Thailand   

                                    location  building_construction_year  \
0               Mediterranean Region, Turkey                         NaN   
1  Kalkan, Mediterranean Region, Kas, Turkey                      2021.0   
2      Mediterranean Region, Antalya, Turkey                         NaN   
3      Chon Buri Province, Pattaya, Thailand                      2020.0   
4      Chon Buri Province, Pattaya, Thailand                      2026.0   

   building_total_floors  apartment_floor  apartment_rooms  \
0                    5.0              1.0              3